# Vehicle Specification Extraction Pipeline
### Assignment Submission — Predii AI Engineering

This notebook demonstrates an end-to-end Retrieval-Augmented Generation (RAG) pipeline for extracting structured mechanical specifications (torque values, fluid capacities, pressures) from automotive service manuals (`sample-service-manual 1.pdf`).

## 1. Setup & Environment

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path
sys.path.append('..')

from src.config import default_config
from src.parser.pdf_loader import PDFLoader
from src.chunking.chunker import SemanticChunker
from src.indexing.embedder import TextEmbedder
from src.indexing.vector_store import VectorStore
from src.extraction.extractor import SpecificationExtractor
from src.utils.exporter import export_to_json, export_to_csv

print("Modules imported successfully!")

## 2. PDF Parsing & Text Cleaning
Using PyMuPDF to parse pages and normalize typographical ligatures (e.g., `\ufb01` -> `fi`).

In [ ]:
pdf_path = default_config.pdf_path
with PDFLoader(pdf_path) as loader:
    print(f"Total Pages in Manual: {loader.total_pages}")
    sample_page = loader.extract_page_text(1)
    print("--- Page 1 Excerpt ---")
    print(sample_page['text'][:300])

## 3. Structure-Aware Chunking
Preserves tabular specification rows and attaches section metadata.

In [ ]:
# Load full manual or cached index
cache_file = default_config.cache_dir / "service_manual_index.pkl"
if cache_file.exists():
    vector_store = VectorStore.load(cache_file)
    print(f"Loaded cached vector store with {len(vector_store.chunks)} chunks.")
else:
    with PDFLoader(pdf_path) as loader:
        pages = loader.load_pages()
    chunker = SemanticChunker(chunk_size=800, chunk_overlap=150)
    chunks = chunker.chunk_document(pages)
    embedder = TextEmbedder()
    vector_store = VectorStore(embedder=embedder)
    vector_store.build_index(chunks)
    vector_store.save(cache_file)
    print(f"Created and cached {len(chunks)} chunks from 852 pages.")

## 4. Retrieval & Specification Extraction

In [ ]:
extractor = SpecificationExtractor(vector_store=vector_store, top_k=5)
query = "Torque for brake caliper bolts"
result = extractor.extract(query)

print(f"Query: {result.query}")
print(f"Status: {result.status} | Time: {result.execution_time_seconds}s | Retrieved Pages: {result.pages_referenced}")

df = pd.DataFrame(result.to_clean_list())
df

## 5. Source Manual Ground-Truth Benchmark Queries
Evaluating 6 test questions directly against known ground-truth specification pages in `sample-service-manual 1.pdf`.

In [ ]:
benchmark_questions = [
    ("Torque for brake caliper anchor plate bolts", "Page 618 / 636: 250 Nm (184 lb-ft)"),
    ("Torque for brake caliper guide pin bolts", "Page 618 / 636: 37 Nm / 33 Nm"),
    ("Stabilizer bar link nuts torque", "Page 24: 70 Nm (52 lb-ft)"),
    ("Lower ball joint nut torque", "Page 24: 175 Nm (129 lb-ft)"),
    ("Tie-rod end nut torque", "Page 24: 115 Nm (85 lb-ft)"),
    ("Brake caliper flow bolt torque", "Page 618 / 636: 35 Nm (26 lb-ft)")
]

benchmark_results = []
for q, expected in benchmark_questions:
    res = extractor.extract(q)
    specs_summary = ", ".join([f"{s.component}: {s.value} {s.unit}" for s in res.specifications[:2]]) or "None"
    benchmark_results.append({
        "Question": q,
        "Expected Ground Truth": expected,
        "Pages Referenced": str(res.pages_referenced[:3]),
        "Extracted Specifications": specs_summary,
        "Status": "MATCH" if len(res.specifications) > 0 else "EMPTY"
    })

pd.DataFrame(benchmark_results)

## 6. Output Export
Exporting structured results to JSON and CSV matching the assignment specification.

In [ ]:
json_file = export_to_json(result, "../outputs/sample_notebook_output.json")
csv_file = export_to_csv(result, "../outputs/sample_notebook_output.csv")
print(f"Saved JSON to: {json_file}")
print(f"Saved CSV to: {csv_file}")